<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M04/M04_Lab1_LangChain_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🔗 M04 Lab 1 — LangChain Basics</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐⭐ Difficulty: Intermediate &nbsp;|&nbsp; ⏱ Time: ~15 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Install and connect <b>LangChain</b> to OpenAI</li>
    <li>Create <b>prompt templates</b> with dynamic variables</li>
    <li>Understand <b>LCEL</b> (LangChain Expression Language) pipe syntax</li>
    <li>Add <b>conversation memory</b> to chains</li>
    <li>Build a <b>multi-step chain</b> that summarizes, translates, and formats</li>
  </ol>
</div>

## 📦 Setup

Before we start, let's install the required packages and set up our API connection.

> **🔑 API Key Setup:** Go to the **key icon** (🔑) in the left sidebar of Colab → click **"Add a secret"** → Name: `OPENAI_API_KEY` → Value: your key → Toggle **"Notebook access"** ON.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai langchain langchain-openai
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
from dads5250 import setup_openai, show_response, show_expected, show_info, quiz

# This reads your OPENAI_API_KEY from Colab Secrets automatically
client = setup_openai()

# Also set the environment variable for LangChain
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ Why LangChain?</h2>
</div>

In M01-M03, we used the **raw OpenAI API** directly. That works great for simple calls, but real applications need:

- **Prompt templates** that reuse logic with different inputs
- **Chains** that connect multiple LLM calls together
- **Memory** so conversations persist across calls
- **Integrations** with vector stores, tools, and agents

**LangChain** is the most popular framework for building LLM applications. Think of it as the **Express.js of AI** — it doesn't replace the raw API, it makes building on top of it much faster.

| Raw API | LangChain |
|---------|----------|
| Manual message formatting | Prompt templates with variables |
| One call at a time | Chain multiple calls together |
| No memory | Built-in conversation memory |
| Manual parsing | Output parsers (JSON, lists, etc.) |

In [ ]:
# ============================================================
# 🔗 Connect to OpenAI via LangChain
# ============================================================
from langchain_openai import ChatOpenAI

# Create a LangChain LLM object — same model, but wrapped in LangChain
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.7)

# Simple invocation — just like the raw API, but cleaner
response = llm.invoke("What is LangChain in one sentence?")
print(f"🤖 {response.content}")
print(f"\n📊 Tokens: {response.usage_metadata}")

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> <code>ChatOpenAI</code> is a thin wrapper around the OpenAI API. You still use the same model names, same API key — LangChain just adds convenience methods on top.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Prompt Templates</h2>
</div>

Instead of manually formatting strings every time, LangChain provides **PromptTemplate** — reusable prompts with variables.

Think of it like a **function**: you define the template once, then call it with different inputs.

In [ ]:
# ============================================================
# 📝 Prompt Templates with Variables
# ============================================================
from langchain_core.prompts import ChatPromptTemplate

# Define a reusable template with {variables}
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Explain concepts clearly in 2-3 sentences."),
    ("user", "Explain {concept} to a {audience}.")
])

# Use it with different inputs
examples = [
    {"domain": "supply chain", "concept": "the bullwhip effect", "audience": "business student"},
    {"domain": "machine learning", "concept": "gradient descent", "audience": "10-year-old"},
    {"domain": "finance", "concept": "compound interest", "audience": "college freshman"},
]

for ex in examples:
    # Format the prompt with variables
    formatted = prompt.format_messages(**ex)
    response = llm.invoke(formatted)
    print(f"\n{'='*60}")
    print(f"📌 {ex['concept']} → {ex['audience']}")
    print(f"{'='*60}")
    print(f"🤖 {response.content}")

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> Prompt templates separate <b>logic</b> from <b>data</b>. You define the structure once and reuse it — just like functions in programming. This is essential for production apps where the same prompt runs thousands of times with different inputs.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ LCEL — The Pipe Syntax</h2>
</div>

**LCEL** (LangChain Expression Language) is LangChain's way of composing components using the **pipe operator** `|`.

```python
chain = prompt | llm | output_parser
```

This reads like a pipeline: **prompt** → feeds into → **llm** → feeds into → **output_parser**

It's inspired by Unix pipes (`cat file | grep error | wc -l`) — each step takes input from the previous step.

In [ ]:
# ============================================================
# 🔗 LCEL: prompt | llm | output_parser
# ============================================================
from langchain_core.output_parsers import StrOutputParser

# Define the chain using LCEL pipe syntax
chain = prompt | llm | StrOutputParser()

# Now invoke it — clean and simple!
result = chain.invoke({
    "domain": "data science",
    "concept": "overfitting",
    "audience": "MBA student"
})

print(f"🤖 {result}")
print(f"\n✅ Type: {type(result).__name__} — StrOutputParser gave us a clean string!")

In [ ]:
# ============================================================
# 🔗 LCEL: A More Practical Chain
# ============================================================

# Chain that generates a product description
product_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a marketing copywriter. Write compelling product descriptions."),
    ("user", "Write a {length} product description for: {product}. Target audience: {audience}.")
])

product_chain = product_prompt | llm | StrOutputParser()

# Generate descriptions for different products
products = [
    {"product": "AI-powered inventory optimizer", "audience": "supply chain managers", "length": "2-sentence"},
    {"product": "smart coffee mug with temperature control", "audience": "tech enthusiasts", "length": "3-sentence"},
]

for p in products:
    desc = product_chain.invoke(p)
    print(f"\n📦 {p['product'].title()}")
    print(f"   {desc}\n")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ Conversation Memory</h2>
</div>

Remember from M01: LLMs are **stateless**. In the raw API, we manually managed message history. LangChain provides **memory** objects that handle this automatically.

**ConversationBufferMemory** stores the full conversation and injects it into each new call.

In [ ]:
# ============================================================
# 🧠 Conversation Memory
# ============================================================
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Prompt with a placeholder for chat history
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor. Keep answers concise (2-3 sentences)."),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{input}")
])

# Create the base chain
base_chain = memory_prompt | llm | StrOutputParser()

# Store for chat histories (keyed by session_id)
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Wrap with message history
chain_with_memory = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# Simulate a multi-turn conversation
config = {"configurable": {"session_id": "student-1"}}

questions = [
    "What is the bullwhip effect in supply chain?",
    "What causes it?",
    "How can AI help reduce it?"   # References earlier context!
]

for q in questions:
    reply = chain_with_memory.invoke({"input": q}, config=config)
    print(f"🧑 Student: {q}")
    print(f"🤖 Tutor:   {reply}\n")

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> Notice how the tutor understood "it" in question 2 and "it" in question 3 — that's the memory working! Each reply is stored and passed back in the next call. Without memory, the model would have no idea what "it" refers to.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">5️⃣ Multi-Step Chains</h2>
</div>

The real power of LCEL is **chaining multiple LLM calls** together. Each step's output becomes the next step's input.

Let's build a chain that:
1. **Summarizes** a long text
2. **Translates** the summary to another language
3. **Formats** the result as a professional report

In [ ]:
# ============================================================
# 🔗 Multi-Step Chain: Summarize → Translate → Format
# ============================================================
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Step 1: Summarize
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following text in exactly 2 sentences."),
    ("user", "{text}")
])
summarize_chain = summarize_prompt | llm | StrOutputParser()

# Step 2: Translate
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate the following text to {language}. Keep it professional."),
    ("user", "{summary}")
])
translate_chain = translate_prompt | llm | StrOutputParser()

# Step 3: Format
format_prompt = ChatPromptTemplate.from_messages([
    ("system", "Format the following text as a professional email snippet with a subject line."),
    ("user", "{translated}")
])
format_chain = format_prompt | llm | StrOutputParser()

# --- Run the full pipeline ---
long_text = """
The global supply chain faced unprecedented disruptions in 2024 due to a combination 
of factors including geopolitical tensions, climate-related events, and shifting consumer 
demand patterns. Companies that had invested in AI-powered demand forecasting and 
digital twin technology were able to adapt more quickly, reducing stockouts by an 
average of 34%. Meanwhile, traditional firms relying on manual forecasting methods 
experienced significant inventory imbalances, with excess inventory costs rising by 22%. 
Experts recommend that companies accelerate their digital transformation initiatives, 
particularly in areas of predictive analytics and real-time visibility across the supply chain.
"""

# Step 1: Summarize
summary = summarize_chain.invoke({"text": long_text})
print("📋 Step 1 — Summary:")
print(f"   {summary}\n")

# Step 2: Translate
translated = translate_chain.invoke({"summary": summary, "language": "Spanish"})
print("🌍 Step 2 — Translation (Spanish):")
print(f"   {translated}\n")

# Step 3: Format
formatted = format_chain.invoke({"translated": translated})
print("✉️ Step 3 — Formatted Email:")
print(f"   {formatted}")

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> Multi-step chains let you break complex tasks into simple, testable steps. Each step does one thing well. This is the foundation of all LangChain applications — and it scales to agents, RAG, and more.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "What does the LCEL pipe syntax `prompt | llm | parser` represent?",
        "options": [
            "Three separate API calls running in parallel",
            "A pipeline where each component's output feeds into the next",
            "A conditional statement — if prompt fails, try llm, then parser",
            "A way to choose between prompt, llm, or parser"
        ],
        "answer": 1,
        "explanation": "LCEL uses the pipe operator to create a sequential pipeline: the prompt formats the input, the LLM generates a response, and the parser extracts the final output. Each step feeds into the next."
    },
    {
        "q": "Why is conversation memory important in LangChain?",
        "options": [
            "It reduces API costs by caching responses",
            "It automatically stores the full chat history and injects it into new calls so the LLM has context",
            "It saves conversations to a database for compliance",
            "It allows the LLM to learn from past conversations permanently"
        ],
        "answer": 1,
        "explanation": "Since LLMs are stateless, memory objects automatically track the conversation and include it in each new call. This gives the illusion of the model 'remembering' previous exchanges."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Build a Summarize → Translate → Format Chain

Complete the code below to build your own multi-step chain. Replace each `-----` with the correct value.

**Your chain should:**
1. Summarize a news article in 1 sentence
2. Translate the summary to French
3. Format it as a tweet (under 280 characters)

In [ ]:
# ============================================================
# 🔧 Exercise: Build Your Own Multi-Step Chain
# ============================================================
# Replace each ----- with the correct value

# Step 1: Summarize prompt
my_summarize = ChatPromptTemplate.from_messages([
    ("-----", "Summarize in exactly 1 sentence."),   # What role is this?
    ("user", "{text}")
])

# Step 2: Translate prompt
my_translate = ChatPromptTemplate.from_messages([
    ("system", "Translate to French."),
    ("user", "{-----}")                               # What variable holds the summary?
])

# Step 3: Format as tweet
my_format = ChatPromptTemplate.from_messages([
    ("system", "Format as a tweet under 280 characters. Add relevant emojis."),
    ("user", "{translated}")
])

# Create individual chains
step1 = my_summarize | ----- | StrOutputParser()      # What goes after the pipe?
step2 = my_translate | llm | -----()                   # What parser extracts a string?
step3 = my_format | llm | StrOutputParser()

# Run the pipeline
article = """
Artificial intelligence is transforming healthcare delivery, with new studies showing 
that AI-assisted diagnostic tools can detect certain cancers up to 20% earlier than 
traditional screening methods. Major hospitals are now integrating these tools into 
their radiology departments, though concerns about data privacy and algorithmic 
bias remain significant barriers to widespread adoption.
"""

s = step1.invoke({"text": article})
print(f"📋 Summary: {s}\n")

t = step2.invoke({"summary": s})
print(f"🇫🇷 French: {t}\n")

tweet = step3.invoke({"translated": t})
print(f"🐦 Tweet: {tweet}")
print(f"   ({len(tweet)} characters)")

**📝 Your Observations:** *(double-click to edit this cell)*

1. Did each step produce a reasonable output? Was any information lost between steps? _[Your answer]_

2. How does the tweet compare to the original article? Is the essence preserved? _[Your answer]_

3. What would happen if you changed the translation language to Japanese? Would the tweet still be under 280 characters? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try building these chains on your own:</p>
  <ol style="font-size: 14px;">
    <li><b>Research chain:</b> Generate a topic outline → Expand each point → Combine into an essay</li>
    <li><b>Code review chain:</b> Explain code → Identify bugs → Suggest improvements</li>
    <li><b>Memory challenge:</b> Create a conversation where you ask 5 follow-up questions — does memory hold up?</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 1 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You've learned LangChain basics — prompt templates, LCEL pipes, memory, and multi-step chains.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><code style="color: #90caf9;">ChatPromptTemplate</code> — reusable prompts with variables</li>
      <li><code style="color: #90caf9;">prompt | llm | parser</code> — LCEL pipe syntax for clean chains</li>
      <li><code style="color: #90caf9;">RunnableWithMessageHistory</code> — automatic conversation memory</li>
      <li>Multi-step chains break complex tasks into simple, testable steps</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M04 Lab 2: Beer Game Supply Chain Simulation</p>
</div>